# Lab 4: Fuzzy Logic and Centre of Gravity (COG) Defuzzification

**Module:** Artificial Intelligence  
**Topic:** Fuzzy Logic

---

### Learning Objectives

By the end of this lab, you will be able to:

1. Understand what fuzzy sets and membership functions are
2. Implement simple triangular and trapezoidal membership functions from scratch
3. Apply fuzzy aggregation (union/intersection)
4. Implement the **Centre of Gravity (COG)** defuzzification method from scratch
5. Visualise membership functions and the COG result

---

## Background

### Fuzzy Logic

Unlike classical (crisp) logic where a value is either **true (1)** or **false (0)**, fuzzy logic allows partial membership. A value can belong to a set with a degree between 0 and 1.

### Membership Functions

A **membership function** μ(x) maps an input value x to a membership degree in [0, 1].

Common shapes:
- **Triangular**: rises linearly to a peak then falls
- **Trapezoidal**: rises to a flat top then falls

### Centre of Gravity (COG) Defuzzification

After applying fuzzy rules we get an aggregated fuzzy output set. **Defuzzification** converts this back to a single crisp number.

The COG (also called the **centroid**) method computes:

$$\text{COG} = \frac{\sum_{i} x_i \cdot \mu(x_i)}{\sum_{i} \mu(x_i)}$$

where $x_i$ are sample points across the output universe of discourse and $\mu(x_i)$ is the membership value at each point.

---

## Setup

Import the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

---

## Exercise 1: Membership Functions from Scratch

Implement two basic membership functions without any fuzzy-logic libraries.

In [ ]:
def triangular_mf(x, a, b, c):
    """
    Triangular membership function.

    Parameters
    ----------
    x : array-like  Input values.
    a : float       Left foot (membership = 0).
    b : float       Peak      (membership = 1).
    c : float       Right foot (membership = 0).

    Returns
    -------
    numpy array of membership degrees in [0, 1].
    """
    x = np.asarray(x, dtype=float)
    mu = np.zeros_like(x)
    # Rising slope: a -> b
    mask1 = (x >= a) & (x <= b)
    mu[mask1] = (x[mask1] - a) / (b - a)
    # Falling slope: b -> c
    mask2 = (x > b) & (x <= c)
    mu[mask2] = (c - x[mask2]) / (c - b)
    return mu


def trapezoidal_mf(x, a, b, c, d):
    """
    Trapezoidal membership function.

    Parameters
    ----------
    x : array-like  Input values.
    a : float       Left foot  (membership = 0).
    b : float       Left shoulder (membership = 1 starts).
    c : float       Right shoulder (membership = 1 ends).
    d : float       Right foot (membership = 0).

    Returns
    -------
    numpy array of membership degrees in [0, 1].
    """
    x = np.asarray(x, dtype=float)
    mu = np.zeros_like(x)
    # Rising slope: a -> b
    mask1 = (x >= a) & (x < b)
    mu[mask1] = (x[mask1] - a) / (b - a)
    # Flat top: b -> c
    mask2 = (x >= b) & (x <= c)
    mu[mask2] = 1.0
    # Falling slope: c -> d
    mask3 = (x > c) & (x <= d)
    mu[mask3] = (d - x[mask3]) / (d - c)
    return mu


# --- Quick test ---
x_test = np.array([0, 25, 50, 75, 100])
print("Triangular MF (25, 50, 75):", triangular_mf(x_test, 25, 50, 75))
print("Trapezoidal MF (10, 30, 70, 90):", trapezoidal_mf(x_test, 10, 30, 70, 90))

---

## Exercise 2: Visualise Membership Functions

In [ ]:
# Universe of discourse: 0 to 100
x = np.linspace(0, 100, 500)

# Define three output fuzzy sets: Low, Medium, High
mu_low    = trapezoidal_mf(x, 0,  0,  20, 40)
mu_medium = triangular_mf(x,  20, 50, 80)
mu_high   = trapezoidal_mf(x,  60, 80, 100, 100)

plt.figure(figsize=(10, 4))
plt.plot(x, mu_low,    label='Low',    color='blue')
plt.plot(x, mu_medium, label='Medium', color='green')
plt.plot(x, mu_high,   label='High',   color='red')
plt.xlabel('Output value')
plt.ylabel('Membership degree μ(x)')
plt.title('Fuzzy Output Membership Functions')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---

## Exercise 3: Aggregate Fuzzy Output

In a simple fuzzy rule system, each rule fires with a certain **activation strength** (alpha cut).  
The activated fuzzy set for each rule is clipped at its alpha value.  
The final aggregated output is the **union (max)** across all activated sets.

In [ ]:
# Suppose three fuzzy rules fire with these activation strengths:
alpha_low    = 0.2   # Rule 1 fires "Low" with strength 0.2
alpha_medium = 0.7   # Rule 2 fires "Medium" with strength 0.7
alpha_high   = 0.4   # Rule 3 fires "High" with strength 0.4

# Clip each fuzzy set at its activation strength
activated_low    = np.fmin(alpha_low,    mu_low)
activated_medium = np.fmin(alpha_medium, mu_medium)
activated_high   = np.fmin(alpha_high,   mu_high)

# Aggregate: union = element-wise maximum
aggregated = np.fmax(np.fmax(activated_low, activated_medium), activated_high)

plt.figure(figsize=(10, 4))
plt.fill_between(x, aggregated, alpha=0.3, color='purple', label='Aggregated output')
plt.plot(x, activated_low,    '--', color='blue',  label=f'Low (α={alpha_low})')
plt.plot(x, activated_medium, '--', color='green', label=f'Medium (α={alpha_medium})')
plt.plot(x, activated_high,   '--', color='red',   label=f'High (α={alpha_high})')
plt.xlabel('Output value')
plt.ylabel('Membership degree μ(x)')
plt.title('Aggregated Fuzzy Output')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---

## Exercise 4: Centre of Gravity (COG) Defuzzification from Scratch

Implement the COG formula:

$$\text{COG} = \frac{\sum_{i} x_i \cdot \mu(x_i)}{\sum_{i} \mu(x_i)}$$

This is a simple weighted average: each point on the x-axis is weighted by its membership degree.

In [ ]:
def centre_of_gravity(x, mu):
    """
    Compute the Centre of Gravity (COG) defuzzified value.

    Parameters
    ----------
    x  : array-like  Universe of discourse sample points.
    mu : array-like  Membership degrees at each sample point.

    Returns
    -------
    float  The crisp COG value.
    """
    x  = np.asarray(x,  dtype=float)
    mu = np.asarray(mu, dtype=float)

    denominator = np.sum(mu)
    if denominator == 0:
        raise ValueError("Sum of membership degrees is zero – cannot defuzzify.")

    return np.sum(x * mu) / denominator


# Apply COG to the aggregated output
cog_value = centre_of_gravity(x, aggregated)
print(f"COG defuzzified value: {cog_value:.2f}")

---

## Exercise 5: Visualise the COG Result

In [ ]:
plt.figure(figsize=(10, 4))
plt.fill_between(x, aggregated, alpha=0.3, color='purple', label='Aggregated output')
plt.plot(x, aggregated, color='purple')
plt.axvline(x=cog_value, color='black', linestyle='--', linewidth=2,
            label=f'COG = {cog_value:.2f}')
plt.xlabel('Output value')
plt.ylabel('Membership degree μ(x)')
plt.title('COG Defuzzification')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"\nThe system output (crisp value) via COG = {cog_value:.2f}")

---

## Summary

In this lab you:

1. Implemented **triangular** and **trapezoidal** membership functions from scratch using NumPy.
2. Defined fuzzy output sets (Low / Medium / High) over a universe of discourse.
3. Clipped each set by its rule activation strength and **aggregated** them using the union (max) operation.
4. Implemented the **Centre of Gravity (COG)** defuzzification formula from scratch as a simple weighted average.
5. Visualised the aggregated fuzzy set and the resulting crisp COG output value.

**Key formula:**
$$\text{COG} = \frac{\sum_{i} x_i \cdot \mu(x_i)}{\sum_{i} \mu(x_i)}$$